# Module 2 - Data Cleaning and Transformation

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession \
    .builder \
        .appName('Databricks_capstone') \
            .getOrCreate()

In [0]:
storage_account="mystoacckad"
application_id="14a14259-70ba-4c26-a136-262468ab64da"
directory_id="26af9d76-35fe-404a-b312-869c37aec9c7"
container_name="fileshare"

service_credential = dbutils.secrets.get(scope="Secrete-scope-databricks1", key="app-reg-secrets1")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               f"org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", service_credential)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{directory_id}/oauth2/token")

In [0]:
folder_path=f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/"
csv=[file_info.name for file_info in dbutils.fs.ls(folder_path) if (file_info.name).endswith('.csv')]

df_customers_dataset=spark.read.csv(folder_path+csv[0],header=True,inferSchema=True)
df_geolocation_dataset=spark.read.csv(folder_path+csv[1],header=True,inferSchema=True)
df_order_items_dataset=spark.read.csv(folder_path+csv[2],header=True,inferSchema=True)
df_order_payments_dataset=spark.read.csv(folder_path+csv[3],header=True,inferSchema=True)
df_order_reviews_dataset=spark.read.csv(folder_path+csv[4],header=True,inferSchema=True)
df_orders_dataset=spark.read.csv(folder_path+csv[5],header=True,inferSchema=True)
df_products_dataset=spark.read.csv(folder_path+csv[6],header=True,inferSchema=True)
df_sellers_dataset=spark.read.csv(folder_path+csv[7],header=True,inferSchema=True)
df_product_category_name_translation=spark.read.csv(folder_path+csv[8],header=True,inferSchema=True)

In [0]:
from pyspark.sql.functions import *
def missing_values(df, df_name):
    print(f"Missing values in {df_name}: ")
    df.select([count(when(col(c).isNull(),1)).alias(c) for c in df.columns]).show()

In [0]:
missing_values(df_customers_dataset,'Customers')		

In [0]:
missing_values(df_geolocation_dataset,'Geolocation')

In [0]:
missing_values(df_order_items_dataset, 'Order Items')

In [0]:
missing_values(df_order_payments_dataset, 'Order Payments')	

In [0]:
missing_values(df_order_reviews_dataset, 'Order Reviews')	

In [0]:
missing_values(df_orders_dataset, 'Orders')			

In [0]:
missing_values(df_products_dataset, 'Products')

In [0]:
missing_values(df_sellers_dataset, 'Sellers')	

In [0]:
missing_values(df_product_category_name_translation, 'Product Category name Translation')

## Handling Missing values
### 1. Drop Missing Values (for non critical columns)
### 2. Fill Missing Values (for numerical values)
### 3. Impute Missing Values (for continous data)

In [0]:
orders_df_cleaned=df_orders_dataset.na.drop(subset=['order_id','customer_id','order_status'])
orders_df_cleaned.show()

In [0]:
orders_df_cleaned=orders_df_cleaned.fillna({'order_delivered_customer_date':'9999-12-31'})
missing_values(orders_df_cleaned, 'Orders')	

## Impute Missing Values

In [0]:
payments